# TrainLM: start a from-scratch language-model run

Run the notebook from top to bottom. TrainLM prepares the tokenizer and packed data, initializes a model from random weights, trains, evaluates, saves checkpoints, exports a Hugging Face model, and writes a run report. You do not write the minibatch loop or assemble the report yourself.

Setup installs missing `datasets` and `tokenizers` extras, stops if an installed version is incompatible, and never requests PyTorch or TorchTPU from pip.

Choose `quick_smoke` (10,000 tokens) to validate the workflow, `starter_pretraining` (1,000,000 tokens) for a longer training run, or `custom` to set your own token target. The smoke test does not measure model quality. A one-million-token run is a starting budget, not a convergence guarantee. Native TorchTPU training remains experimental.


In [ ]:
#@title 0. Set up TrainLM
from importlib import metadata

from pathlib import Path
import subprocess
import sys
import tempfile

from packaging.requirements import Requirement
from packaging.version import InvalidVersion, Version
repo_root = Path(tempfile.mkdtemp(prefix="trainlm-pretraining-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
pretraining_requirements = (
    Requirement("datasets>=3,<5"),
    Requirement("tokenizers>=0.21,<1"),
)
missing_pretraining_requirements = []
incompatible_pretraining_requirements = []
for requirement in pretraining_requirements:
    try:
        installed = Version(metadata.version(requirement.name))
    except metadata.PackageNotFoundError:
        missing_pretraining_requirements.append(str(requirement))
        continue
    except InvalidVersion:
        incompatible_pretraining_requirements.append(
            f"{requirement.name} has invalid installed metadata"
        )
        continue
    if installed not in requirement.specifier:
        incompatible_pretraining_requirements.append(
            f"{requirement.name} {installed} does not satisfy "
            f"{requirement.specifier}"
        )
if incompatible_pretraining_requirements:
    details = "; ".join(incompatible_pretraining_requirements)
    raise RuntimeError(
        f"Incompatible pretraining packages: {details}. "
        "Start with a fresh runtime or resolve these versions first; "
        "the setup cell leaves installed versions unchanged."
    )
if missing_pretraining_requirements:
    print("Installing missing optional pretraining packages...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--quiet", "--disable-pip-version-check",
            *missing_pretraining_requirements,
        ],
        check=True,
    )
for requirement in pretraining_requirements:
    installed = Version(metadata.version(requirement.name))
    if installed not in requirement.specifier:
        raise RuntimeError(
            f"Could not satisfy {requirement.name} {requirement.specifier}."
        )
    print(f"{requirement.name}=={installed}")

source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()


## 1. Choose the run

Choose a training preset below. `quick_smoke` checks that setup, training, evaluation, and export work; it is not useful-model training. `starter_pretraining` sets a one-million-token budget. Select `custom` to enter another budget.

In Colab, set `RUN_DIR` under mounted Google Drive to keep checkpoints across runtime resets. Kaggle uses `/kaggle/working` by default.

To resume from a downloaded run report, restore the entire run directory, enable `RESUME_FROM_REPORT`, then run all cells. For an ordinary resume in the same source/runtime, set `RESUME_FROM_CHECKPOINT` to `latest`. Turn on budget extension only when increasing the token target.


In [ ]:
#@title 1. Choose your run
from pathlib import Path
import json
import re
import subprocess
import sys

RUN_DIR = "/content/trainlm-pretraining" #@param {type:"string"}
TRAINING_PRESET = "quick_smoke (10k tokens)" #@param ["quick_smoke (10k tokens)", "starter_pretraining (1M tokens)", "custom target"]
CUSTOM_TARGET_SUPERVISED_TOKENS = 1_000_000 #@param {type:"integer"}
SEQUENCE_LENGTH = 128 #@param {type:"integer"}
RESUME_FROM_REPORT = False #@param {type:"boolean"}
RESUME_FROM_CHECKPOINT = "" #@param {type:"string"}
EXTEND_TRAINING_BUDGET = False #@param {type:"boolean"}

preset_token_targets = {
    "quick_smoke (10k tokens)": 10_000,
    "starter_pretraining (1M tokens)": 1_000_000,
    "custom target": CUSTOM_TARGET_SUPERVISED_TOKENS,
}
TARGET_SUPERVISED_TOKENS = preset_token_targets.get(TRAINING_PRESET)
if type(TARGET_SUPERVISED_TOKENS) is not int or TARGET_SUPERVISED_TOKENS < 1:
    raise ValueError("The token target must be a positive integer.")

if Path("/kaggle/working").is_dir() and RUN_DIR == "/content/trainlm-pretraining":
    RUN_DIR = "/kaggle/working/trainlm-pretraining"
RUN_DIR = Path(RUN_DIR).expanduser()
RUN_DIR.mkdir(parents=True, exist_ok=True)
report_path = RUN_DIR / "report.json"
restored_run = None

if RESUME_FROM_REPORT:
    if not report_path.is_file():
        raise FileNotFoundError(f"Restore the original run directory: {report_path}")
    saved_report = json.loads(report_path.read_text(encoding="utf-8"))
    saved_revision = saved_report.get("source_revision")
    if not isinstance(saved_revision, str) or not re.fullmatch(r"[0-9a-f]{40}", saved_revision):
        raise ValueError("The report does not contain a valid TrainLM source revision.")
    subprocess.run(
        ["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", saved_revision],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo_root), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
    source_revision = subprocess.check_output(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if source_revision != saved_revision:
        raise RuntimeError("Could not restore the TrainLM revision recorded in report.json.")

sys.path.insert(0, str(repo_root / "src"))
from trainlm import restore_pretraining_report

if RESUME_FROM_REPORT:
    restored_run = restore_pretraining_report(report_path, run_dir=RUN_DIR)
    RESUME_FROM_CHECKPOINT = restored_run.checkpoint_path
    EXTEND_TRAINING_BUDGET = (
        EXTEND_TRAINING_BUDGET or restored_run.extend_training_budget
    )
else:
    checkpoint_value = RESUME_FROM_CHECKPOINT.strip()
    RESUME_FROM_CHECKPOINT = (
        None if not checkpoint_value
        else "latest" if checkpoint_value == "latest"
        else Path(checkpoint_value).expanduser()
    )

if restored_run is not None and not EXTEND_TRAINING_BUDGET:
    print(f"resuming_saved_target_tokens={restored_run.config.max_tokens:,}")
else:
    print(f"training_preset={TRAINING_PRESET} target_tokens={TARGET_SUPERVISED_TOKENS:,}")


### Optional: customize the data or model

Leave these defaults unchanged for the starter run. Use an immutable dataset revision. For another registered Transformers causal-LM architecture, set `MODEL_TYPE` and provide its small-model config fields in `MODEL_CONFIG_OVERRIDES`. The public API validates these values before training.

For larger models, set `GRADIENT_CHECKPOINTING = True` to trade some compute for lower activation memory. Leave it off for the baseline; native TPU memory/performance remains unverified.


In [ ]:
#@title Optional: advanced data and model settings
DATASET_ID = "roneneldan/TinyStories" #@param {type:"string"}
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64" #@param {type:"string"}
TRAIN_SPLIT = "train" #@param {type:"string"}
VALIDATION_SPLIT = "validation" #@param {type:"string"}
TEXT_FIELD = "text" #@param {type:"string"}
MODEL_TYPE = "gpt2" #@param {type:"string"}
MODEL_CONFIG_OVERRIDES = None
GRADIENT_CHECKPOINTING = False #@param {type:"boolean"}


## 2. Train, evaluate, and export

This calls the high-level public API. The API owns preparation, training, evaluation, checkpoint lookup, export, and the versioned report. For advanced control, use `TrainLMTrainer.from_scratch(config)` directly.


In [ ]:
#@title 2. Run TrainLM
from dataclasses import replace
import json
from trainlm import (
    TrainLMPretrainingConfig,
    TrainLMPretrainingError,
    run_pretraining,
)

if restored_run is not None:
    pretraining_config = restored_run.config
    if EXTEND_TRAINING_BUDGET:
        if TARGET_SUPERVISED_TOKENS <= pretraining_config.max_tokens:
            raise ValueError("Set a larger token target to extend this run.")
        pretraining_config = replace(
            pretraining_config, max_tokens=TARGET_SUPERVISED_TOKENS
        )
else:
    pretraining_config = TrainLMPretrainingConfig(
        output_dir=RUN_DIR / "artifacts",
        dataset_id=DATASET_ID,
        dataset_revision=DATASET_REVISION,
        train_split=TRAIN_SPLIT,
        validation_split=VALIDATION_SPLIT,
        text_field=TEXT_FIELD,
        model_type=MODEL_TYPE,
        model_config_overrides=MODEL_CONFIG_OVERRIDES,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        max_tokens=TARGET_SUPERVISED_TOKENS,
        sequence_length=SEQUENCE_LENGTH,
        save_total_limit=3,
        accelerator="native_tpu",
    )

try:
    result = run_pretraining(
        pretraining_config,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
        extend_training_budget=EXTEND_TRAINING_BUDGET,
        show_progress=True,
        source_revision=source_revision,
        report_path=report_path,
    )
except TrainLMPretrainingError as error:
    print(f"report={error.report_path}")
    print(json.dumps(error.report, indent=2))
    raise

print(result.summary())
print(f"report={result.report_path}")
if result.status != "passed":
    raise RuntimeError(f"Run did not meet its target; inspect {result.report_path}.")


## 3. Download the result

The ZIP contains the model export when valid, `report.json`, and a ready-to-run resume example when a checkpoint exists. Keep the original run directory for optimizer state and packed data; the ZIP does not contain the full checkpoint directory.


In [ ]:
#@title 3. Download the model and report
from pathlib import Path
import json
import shutil
from trainlm import bundle_pretraining_artifacts

artifact_bundle = bundle_pretraining_artifacts(report_path)
report = json.loads(report_path.read_text(encoding="utf-8"))
resume_code = (report.get("resume") or {}).get("code")
if resume_code:
    from IPython.display import Code, display
    display(Code(resume_code, language="python"))

if Path("/kaggle/working").is_dir():
    kaggle_bundle = Path("/kaggle/working") / artifact_bundle.name
    if kaggle_bundle.resolve() != artifact_bundle.resolve():
        shutil.copy2(artifact_bundle, kaggle_bundle)
    print(f"artifact_bundle={kaggle_bundle}")
try:
    from google.colab import files
except ImportError:
    print(f"artifact_bundle={artifact_bundle}")
else:
    files.download(str(artifact_bundle))
